# Bidirectional Graph Relations

SurrealEngine supports two complementary patterns for traversing relationships in both directions:

1. **Graph edge traversal** — using `RELATE` (via `RelationDocument`) and traversing with `.out()` and `.in_()` on the ORM QuerySet.
2. **`IncomingReferenceField`** — a native SurrealDB 3.x feature for **record-reference fields** declared with `REFERENCE`. This creates a `COMPUTED <~table` field that live-indexes who is pointing at a given record.

> **Key distinction:** `IncomingReferenceField` only tracks `REFERENCE`-typed fields (normal document fields declared with `reference=True`), **not** `RELATE` graph edges. For `RelationDocument` followers, use `.in_()` traversal (cell 5).


## 1. Setup

In [ ]:
from surrealengine import (
    Document,
    StringField,
    IntField,
    BooleanField,
    RelationDocument,
    ReferenceField,
    create_connection,
    Q,
)
from surrealengine.fields import IncomingReferenceField

conn = create_connection(
    url="ws://db:8000/rpc",
    namespace="test_ns",
    database="bidir_demo",
    username="root",
    password="root",
    make_default=True,
)
await conn.connect()
print("Connected!")

## 2. Define Schema

### Approach A — Graph edges via `RelationDocument`

A `RelationDocument` writes a `RELATE` edge. Both directions are traversable via `.out()` / `.in_()` — no special flag needed.

### Approach B — `IncomingReferenceField` (COMPUTED `<~table`)

Requires:
- A source field typed `record<T>` **with `reference=True`** — SurrealDB adds the `REFERENCE` clause that enables change tracking.
- A `COMPUTED <~source_table` field on the target model — provided by `IncomingReferenceField`.

Below, `Tag.tagged_users` is a `ReferenceField(TaggedUser, reference=True)`, so SurrealDB emits:
```sql
DEFINE FIELD tagged_users ON tag TYPE option<record<tagged_user>> REFERENCE
DEFINE FIELD tagged_in ON tagged_user COMPUTED <~tag
```

In [ ]:
# -----------------------------------------------------------------------
# Approach A — graph edges
# -----------------------------------------------------------------------

class User(Document):
    name = StringField(required=True)
    email = StringField()

    class Meta:
        collection = "user"


class Follows(RelationDocument):
    """A follows relationship. in=follower, out=followee."""
    since_year = IntField(default=2024)
    is_mutual = BooleanField(default=False)

    class Meta:
        collection = "follows"


# -----------------------------------------------------------------------
# Approach B — IncomingReferenceField
#
# TaggedUser.tagged_in is COMPUTED <~tag — it returns all Tag records
# whose `tagged_users` field (declared with reference=True) points here.
# -----------------------------------------------------------------------

class TaggedUser(Document):
    name = StringField(required=True)
    # SurrealDB emits: DEFINE FIELD tagged_in ON tagged_user COMPUTED <~tag
    tagged_in = IncomingReferenceField('Tag')

    class Meta:
        collection = "tagged_user"


class Tag(Document):
    """A tag that links to a TaggedUser via a REFERENCE-tracked field."""
    label = StringField(required=True)
    # reference=True causes SurrealDB to emit REFERENCE, enabling <~tag tracking
    tagged_users = ReferenceField('TaggedUser', reference=True)

    class Meta:
        collection = "tag"


# Register schemas — TaggedUser before Tag so forward ref resolves
await User.create_table()
await Follows.create_table()
await TaggedUser.create_table()
await Tag.create_table()
print("Schema registered.")

## 3. Seed Data

In [ ]:
try:
    await Follows.objects.delete()
    await User.objects.delete()
    await Tag.objects.delete()
    await TaggedUser.objects.delete()
except Exception:
    pass

alice = User(name="Alice", email="alice@example.com")
await alice.save()
bob = User(name="Bob", email="bob@example.com")
await bob.save()
carol = User(name="Carol", email="carol@example.com")
await carol.save()
dave = User(name="Dave", email="dave@example.com")
await dave.save()

print(f"Users: alice={alice.id}, bob={bob.id}, carol={carol.id}, dave={dave.id}")

In [ ]:
ab = await Follows.create_relation(alice, bob,   since_year=2020)
ac = await Follows.create_relation(alice, carol, since_year=2021)
ba = await Follows.create_relation(bob,   alice, since_year=2022, is_mutual=True)
bd = await Follows.create_relation(bob,   dave,  since_year=2023)
cd = await Follows.create_relation(carol, dave,  since_year=2024)

print("Edges created.")
print(f"  Alice -> Bob:   {ab.id}")
print(f"  Alice -> Carol: {ac.id}")
print(f"  Bob   -> Alice: {ba.id}")
print(f"  Bob   -> Dave:  {bd.id}")
print(f"  Carol -> Dave:  {cd.id}")

## 4. Forward Traversal — who does Alice follow?

In [ ]:
alice_following = await User.objects.filter(id=alice.id).out("follows").out(User)
print("Alice follows:", [u.name for u in alice_following])

## 5. Reverse Traversal — who follows Alice? (`.in_()`)

SurrealDB always stores both directions of a `RELATE` edge — no extra schema flag needed.

In [ ]:
alice_followers = await User.objects.filter(id=alice.id).in_("follows").in_(User).all()
print("Followers of Alice:", [u.name for u in alice_followers])

## 6. Approach B — `IncomingReferenceField` and Hydration Depth

`Tag.tagged_users = ReferenceField('TaggedUser', reference=True)` emits:
```sql
DEFINE FIELD tagged_users ON tag TYPE option<record<tagged_user>> REFERENCE
```
SurrealDB live-tracks every `tag` record that points to a `tagged_user`. The `COMPUTED <~tag` field on `TaggedUser` reads this index automatically.

### Hydration depth — how deep does it go?

When you call `await alice_tu.refresh()`, SurrealEngine automatically detects `IncomingReferenceField` fields and issues:
```sql
SELECT * FROM tagged_user:<id> FETCH tagged_in
```
This is **one level deep**. The `Tag` objects inside `tagged_in` are fully hydrated — you get real `Tag` instances with all their fields. However, any `ReferenceField` *on those Tag objects* (like `Tag.tagged_users`) is returned as a `RecordID`, **not** a further hydrated `TaggedUser`.

This is the correct default — it mirrors industry-standard ORM behaviour (Django, SQLAlchemy, MongoEngine). Unlimited deep hydration would:
- Risk infinite recursion on circular references (e.g. Tag → TaggedUser → Tag → ...),
- Silently load large portions of your database graph.

**Rule of thumb:** One level deep by default. Opt-in to deeper loading when you need it.

If you need to go deeper, SurrealDB supports chained `FETCH` natively — `FETCH tagged_in, tagged_in.tagged_users` — and future versions of SurrealEngine will expose this via a `refresh(fetch=[...])` parameter.

In [ ]:
alice_tu = TaggedUser(name="Alice")
await alice_tu.save()
bob_tu = TaggedUser(name="Bob")
await bob_tu.save()

python_tag  = Tag(label="python",   tagged_users=alice_tu)
await python_tag.save()
surreal_tag = Tag(label="surrealdb", tagged_users=bob_tu)
await surreal_tag.save()

# 1-level deep: tagged_in returns fully-hydrated Tag ORM objects
await alice_tu.refresh()
print("Alice is tagged in (hydrated Tag objects):")
for tag in alice_tu.tagged_in:
    print(f"  Tag(label={tag.label!r}, id={tag.id})")

# Accessing a ReferenceField *on* the hydrated Tag returns a RecordID
# (one level deep — not automatically followed further).
first_tag = alice_tu.tagged_in[0]
print(f"\nfirst_tag.tagged_users = {first_tag.tagged_users!r}  ← RecordID, not a TaggedUser")
print("This is intentional: only 1 level is fetched by default.")
print("\nTo go deeper, query the referenced record manually:")
deeper = await TaggedUser.objects.get(id=first_tag.tagged_users)
print(f"  TaggedUser(name={deeper.name!r}, id={deeper.id})")

## 7. Filter by Edge Properties

In [ ]:
recent_edges = await Follows.objects.filter(
    Follows.since_year >= 2022
).all()
print("Follows edges from 2022+:", [(e.since_year, str(e.out)) for e in recent_edges])

## 8. Mutual Follows

In [ ]:
alice_following = await User.objects.filter(id=alice.id).out("follows").out(User)
following_ids = {str(u.id) for u in alice_following}

alice_follower_list = await User.objects.filter(id=alice.id).in_("follows").in_(User).all()
follower_ids = {str(u.id) for u in alice_follower_list}

mutual_ids = following_ids & follower_ids
mutuals = [u for u in alice_following if str(u.id) in mutual_ids]
print("Mutual follows (friends of Alice):", [u.name for u in mutuals])

## 9. Multi-Hop — reachable in two hops

In [ ]:
two_hops = await User.objects.filter(id=alice.id).out("follows").out("follows").out(User)
print("Alice can reach in 2 hops:", [u.name for u in two_hops])

## 10. Cleanup

In [ ]:
await Follows.objects.delete()
await User.objects.delete()
await Tag.objects.delete()
await TaggedUser.objects.delete()
await conn.close()
print("Done.")